In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    t = tmp / name
    if not t.exists(): os.symlink(w, t)
wheels = sorted(str(w) for w in tmp.glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('installed', len(wheels), 'wheels')

In [ ]:
import base64, hashlib, json, shutil, subprocess, zipfile
from pathlib import Path
DATA = next(Path('/kaggle/input').rglob('expected.json')).parent
exp = json.loads((DATA / 'expected.json').read_text())
raw = base64.b64decode(next(DATA.glob('*.zip.b64.txt')).read_text())
zp = Path('/tmp/check.zip'); zp.write_bytes(raw)
print('zip sha256 matches local build:', hashlib.sha256(raw).hexdigest() == exp['zip_sha256'], '|', len(raw), 'bytes')
from adk_submission import compile_submission
from swegemma.context import SwegemmaContext
from swegemma.models import Task, resolve_swegemma_adapter, setup_gemma_model_registry
models = setup_gemma_model_registry()
task = Task(instance_id='d', repo='d/d', base_commit='0'*40, problem_statement='d')
ctx = SwegemmaContext(docker_manager=None, container_id='', problem_statement='d', hints_text='', task=task, repo='d/d', graph_dir='/tmp', embeddings_dir='/tmp')
tools = ctx.create_tools()
ok_all = True
def check(label, d):
    global ok_all
    files = {str(p.relative_to(d)): hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(d).rglob('*') if p.is_file()}
    same = files == exp['files']
    print(f'  [{label}] extracted files:', sorted(files), '| identical to expected contents:', same)
    try:
        agent = compile_submission(submission_dir=Path(d), tool_registry=tools, model_registry=models, adapter_resolver_fn=lambda b, i: resolve_swegemma_adapter(b, i, models))
        print(f'  [{label}] COMPILE OK: {agent.name}'); ok = True
    except Exception as e:
        print(f'  [{label}] COMPILE FAILED: {type(e).__name__}: {str(e)[:300]}'); ok = False
    ok_all = ok_all and same and ok
dA = Path('/tmp/ex_py'); shutil.rmtree(dA, ignore_errors=True); dA.mkdir()
with zipfile.ZipFile(zp) as z:
    print('  stored names (Linux zipfile):', z.namelist()); z.extractall(dA)
check('python-extract', dA)
dB = Path('/tmp/ex_unzip'); shutil.rmtree(dB, ignore_errors=True); dB.mkdir()
r = subprocess.run(['unzip', '-o', str(zp), '-d', str(dB)], capture_output=True, text=True)
print('  unzip rc:', r.returncode, (r.stderr or '').strip()[:200])
check('unzip-extract', dB)
print('\nZIP CHECK RESULT:', 'PASS' if ok_all else 'FAIL')